In [27]:
from google.colab import drive
drive.mount ('/content/drive')
print ('Drive mounted!')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted!


In [28]:
import sys
print(f'Python Version is {sys.version}')
import pandas as pd
import numpy as np

Python Version is 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [29]:
file_path='/content/drive/MyDrive/EmergencyTriageDataset_Reduced_Dirty.csv'
df=pd.read_csv(file_path)
df.head(10)

,ID,Age,Gender,GCS,SBP,DBP,MAP,pulse,Temp,RR,Fio2
0,1,34,0,15.0,93,67.0,75.67,128.0,36.8,14.0,21.0
1,2,20,Male,15.0,130,90.0,103.33,80.0,37.0,16.0,21.0
2,3,77,Female,14.0,163,105.0,124.33,92.0,36.8,18.0,21.0
3,4,23,0,8.0,100,60.0,73.33,100.0,37.0,12.0,100.0
4,5,86,FEMALE,15.0,150,90.0,110.00,85.0,37.0,19.0,21.0
5,6,42,Male,15.0,100,60.0,73.33,99.0,37.0,20.0,21.0
6,7,75,Female,15.0,120,80.0,93.33,99.0,37.0,25.0,21.0
7,8,25,Male,15.0,100,50.0,66.67,85.0,37.0,25.0,21.0
8,9,67,0,15.0,110,70.0,83.33,78.0,37.0,16.0,21.0
9,11,82,Female,15.0,153,82.0,105.67,130.0,37.0,19.0,21.0


In [30]:
gender_map = {'Male': 1, 'MALE': 1, '1': 1, 'Female': 0, 'FEMALE': 0, '0': 0}
df['Gender'] = df['Gender'].map(gender_map)

print(f"Dataset: {df.shape[0]} rows x {df.shape[1]} columns")

Dataset: 2205 rows x 11 columns


In [31]:
print("=== head() — first 5 rows ===")
print(df.head())

=== head() — first 5 rows ===
   ID  Age  Gender   GCS  SBP    DBP     MAP  pulse  Temp    RR   Fio2
0   1   34       0  15.0   93   67.0   75.67  128.0  36.8  14.0   21.0
1   2   20       1  15.0  130   90.0  103.33   80.0  37.0  16.0   21.0
2   3   77       0  14.0  163  105.0  124.33   92.0  36.8  18.0   21.0
3   4   23       0   8.0  100   60.0   73.33  100.0  37.0  12.0  100.0
4   5   86       0  15.0  150   90.0  110.00   85.0  37.0  19.0   21.0


In [32]:
print("=== tail() — last 5 rows (catches truncation issues) ===")
print(df.tail())

=== tail() — last 5 rows (catches truncation issues) ===
        ID  Age  Gender   GCS  SBP    DBP     MAP  pulse  Temp    RR  Fio2
2200  2380   65       1  14.0  170   50.0   90.00   65.0  37.0  22.0  21.0
2201  2381   84       0  15.0  180  120.0  140.00   85.0  36.1  18.0  21.0
2202  2382   78       0  15.0  132   78.0   96.00  100.0  37.9  16.0  21.0
2203  2383   65       1  15.0  140   80.0  100.00   85.0  36.4  16.0  21.0
2204  2384   78       0  15.0  145   90.0  108.33   75.0  37.6  20.0  21.0


In [33]:
print("=== info() — column types and non-null counts ===")
df.info()

=== info() — column types and non-null counts ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2205 entries, 0 to 2204
Data columns (total 11 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   ID      2205 non-null   int64  
 1   Age     2205 non-null   int64  
 2   Gender  2205 non-null   int64  
 3   GCS     2183 non-null   object 
 4   SBP     2205 non-null   object 
 5   DBP     2183 non-null   float64
 6   MAP     2183 non-null   float64
 7   pulse   2183 non-null   object 
 8   Temp    2183 non-null   object 
 9   RR      2183 non-null   float64
 10  Fio2    2183 non-null   float64
dtypes: float64(4), int64(3), object(4)
memory usage: 189.6+ KB


In [34]:
print("=== describe() — statistics for numeric columns only ===")
print(df.describe())

=== describe() — statistics for numeric columns only ===
                ID          Age       Gender          DBP          MAP  \
count  2205.000000  2205.000000  2205.000000  2183.000000  2183.000000   
mean   1154.987755    61.829478     0.533333    77.479615    93.912277   
std     677.167364    18.485363     0.499001    16.713289    19.006296   
min       1.000000    18.000000     0.000000    30.000000    39.330000   
25%     577.000000    50.000000     0.000000    70.000000    82.500000   
50%    1135.000000    64.000000     1.000000    78.000000    93.330000   
75%    1703.000000    77.000000     1.000000    87.000000   103.330000   
max    2384.000000    98.000000     1.000000   173.000000   185.000000   

                RR         Fio2  
count  2183.000000  2183.000000  
mean     20.262254    25.019698  
std       5.742333    10.144288  
min      12.000000    21.000000  
25%      17.000000    21.000000  
50%      18.000000    21.000000  
75%      21.000000    21.000000  
max 

### **Preparing SBP for MAP Cleaning**

SBP is the peak pressure in the arteries when the heart beats — the top
number in a reading like 120/80. It is one of the fastest triage signals
for identifying a patient in danger. An SBP below 90 mmHg indicates
hypotension; below 70 mmHg suggests the patient may be in shock.

**Cleaning decisions:**
The column was stored as strings (object dtype), requiring conversion to
numeric before any analysis could be done. 44 values of 30 and 500 were
identified — these are data entry errors, not outliers. A real SBP of 30
is not survivable outside a resuscitation bay, and 500 exceeds what human
vasculature can physically generate. These were replaced with NaN.

After removing the invalid values, the distribution was approximately
normal (mean 126.68, median 125.00 — difference of 1.68 mmHg), so we
imputed with the **mean**.

In [35]:
# SBP = Systolic Blood Pressure. Valid range: 50–250 mmHg.
# Extremely low SBP (<70) = shock risk. Extremely high (>200) = hypertensive crisis.

print("SBP unique values (sample):", df['SBP'].unique()[:15])
print("SBP dtype:", df['SBP'].dtype)

SBP unique values (sample): ['93' '130' '163' '100' '150' '120' '110' '153' '152' '186' '107' '80'
 '75' '58' '140']
SBP dtype: object


In [36]:
# 1. Convert to numeric — non-numeric entries become NaN
df['SBP'] = pd.to_numeric(df['SBP'], errors='coerce')
print("After type conversion:")
print(df['SBP'].describe())

After type conversion:
count    2183.000000
mean      128.388456
std        43.881583
min        30.000000
25%       110.000000
50%       125.000000
75%       140.000000
max       500.000000
Name: SBP, dtype: float64


In [37]:
# 2. Identify out-of-range values
invalid_sbp = df[(df['SBP'] < 50) | (df['SBP'] > 250)]
print(f"Out-of-range SBP rows: {len(invalid_sbp)}")
print(invalid_sbp[['ID', 'SBP']].head())

Out-of-range SBP rows: 44
      ID    SBP
29    32   30.0
111  123  500.0
168  183  500.0
231  247   30.0
275  291  500.0


In [38]:
# 3. Replace out-of-range with NaN, then impute with median
df.loc[(df['SBP'] < 50) | (df['SBP'] > 250), 'SBP'] = np.nan
sbp_median = df['SBP'].median()
df['SBP'] = df['SBP'].fillna(sbp_median)

print(f"SBP after cleaning: min={df['SBP'].min()}, max={df['SBP'].max()}, NaNs={df['SBP'].isnull().sum()}")

SBP after cleaning: min=55.0, max=250.0, NaNs=0


### **Preparing DBP for MAP Cleaning**

Since the assigned column is `MAP`, we first prepared `DBP` because MAP is calculated using both systolic and diastolic blood pressure. `DBP` stands for diastolic blood pressure, which is the lower number in a blood pressure reading.

The valid DBP range used for this task is `30-150 mmHg`. Values outside this range were treated as invalid and converted to `NaN` so they would not be used directly in the MAP calculation.

After invalid and missing DBP values were identified, I used **median imputation** to fill the missing DBP entries. The median was selected because blood pressure values can contain outliers, and the median is less affected by extreme values than the mean. This allowed the DBP column to remain usable for the later MAP calculation while reducing the influence of unusually high or low values.

In [39]:
# Cleaning DBP as preparation for MAP calculation

# Inspecting DBP before cleaning
print("DBP dtype:", df["DBP"].dtype)
print("DBP unique values sample:", df["DBP"].unique()[:20])
print("DBP missing values before cleaning:", df["DBP"].isnull().sum())

DBP dtype: float64
DBP unique values sample: [ 67.  90. 105.  60.  80.  50.  70.  82.  89. 100.  84.  40.  61.  32.
  53.  63.  nan  76.  75.  48.]
DBP missing values before cleaning: 22


In [40]:
# Converting DBP to numeric
df["DBP"] = pd.to_numeric(df["DBP"], errors="coerce")

print("\nDBP dtype after conversion:", df["DBP"].dtype)
print("DBP missing values after conversion:", df["DBP"].isnull().sum())


DBP dtype after conversion: float64
DBP missing values after conversion: 22


In [41]:
# Flagging invalid DBP values
# Valid DBP range: 30-150 mmHg
invalid_dbp = df[(df["DBP"] < 30) | (df["DBP"] > 150)]

print(f"\nDBP values outside valid range (30-150): {len(invalid_dbp)} rows")
print(invalid_dbp[["ID", "DBP"]].head(10))


DBP values outside valid range (30-150): 3 rows
        ID    DBP
375    395  173.0
951    981  160.0
1468  1510  160.0


In [42]:
# Replacing invalid DBP values with NaN
df.loc[(df["DBP"] < 30) | (df["DBP"] > 150), "DBP"] = np.nan

print("\nDBP missing values after range filtering:", df["DBP"].isnull().sum())


DBP missing values after range filtering: 25


In [43]:
# Imputing missing DBP values using the median
# Median is used because blood pressure values can contain outliers,
# and the median is less affected by extreme values than the mean.

dbp_median = df["DBP"].median()

df["DBP"] = df["DBP"].fillna(dbp_median)

print(f"\nMedian DBP used for imputation: {dbp_median}")
print("DBP missing values after median imputation:", df["DBP"].isnull().sum())

print("\nDBP summary after imputation:")
print(df["DBP"].describe())

print("DBP missing values after range filtering:", df["DBP"].isnull().sum())


Median DBP used for imputation: 78.0
DBP missing values after median imputation: 0

DBP summary after imputation:
count    2205.000000
mean       77.367347
std        16.316160
min        30.000000
25%        70.000000
50%        78.000000
75%        87.000000
max       150.000000
Name: DBP, dtype: float64
DBP missing values after range filtering: 0


### **Finalising MAP Cleaning**

MAP represents the average pressure driving blood to organs across a full cardiac cycle. A healthy MAP is typical within the range of 70 to 100 mmHg, and a map below that is a clinical threshold for organ hypoperfusion.

How do we calculate MAP?/Why we recalculate instead of using the filter?
MAP is not directly measured — it is calculated from the (above) SBP and DBP rating:

MAP = (SBP + 2 * DBP) /3

Before cleaning, 43 rows had MAP values that didn't match the formula, because MAP had been computed from the dirty SBP data. Simple filtration would leave those "silently" wrong values in place. However, recalculating from the cleaness SBP and DBP guarantees consistency/correctness.

In [44]:
# Recalculate MAP from cleaned SBP and DBP instead of filtering the existing column
# — 43 rows had MAP corrupted by dirty SBP values before cleaning
df['MAP'] = ((df['SBP'] + 2 * df['DBP']) / 3).round(2)

print('MAP after recalculation:')
print(df['MAP'].describe())

MAP after recalculation:
count    2205.000000
mean       93.787079
std        18.547074
min        39.330000
25%        83.330000
50%        93.330000
75%       103.330000
max       173.330000
Name: MAP, dtype: float64


In [45]:
out_of_range_map = df[(df['MAP'] < 40) | (df['MAP'] > 180)]
print(f'MAP values outside 40–180 mmHg: {len(out_of_range_map)}')

if not out_of_range_map.empty:
    print(out_of_range_map[['ID', 'SBP', 'DBP', 'MAP']])
    print()
    print('Both SBP and DBP are within valid ranges — this is a critically ill')
    print('patient (severe hypotension), not a data error. Preserved intentionally.')

MAP values outside 40–180 mmHg: 1
        ID   SBP   DBP    MAP
1710  1762  58.0  30.0  39.33

Both SBP and DBP are within valid ranges — this is a critically ill
patient (severe hypotension), not a data error. Preserved intentionally.


In [46]:
map_median = df['MAP'].median()
df['MAP'] = df['MAP'].fillna(map_median)

print(f'MAP NaNs after fallback imputation: {df["MAP"].isnull().sum()}')
print(f'Imputed with mean: {map_median:.2f} mmHg')

MAP NaNs after fallback imputation: 0
Imputed with mean: 93.33 mmHg


### **Final Sanity Check**

As to check that the values are consistent and correctly imputated.

In [47]:
checks = [('SBP', 50, 250), ('DBP', 30, 150), ('MAP', 40, 180)]

for col, lo, hi in checks:
    n_invalid = ((df[col] < lo) | (df[col] > hi)).sum()
    n_null    = df[col].isnull().sum()
    flag = '✓' if (n_invalid == 0 and n_null == 0) else '⚠'
    print(f'{flag} {col}: range {df[col].min():.1f}–{df[col].max():.1f} | '
          f'out-of-range: {n_invalid} | NaNs: {n_null}')

✓ SBP: range 55.0–250.0 | out-of-range: 0 | NaNs: 0
✓ DBP: range 30.0–150.0 | out-of-range: 0 | NaNs: 0
⚠ MAP: range 39.3–173.3 | out-of-range: 1 | NaNs: 0


### **Export**

Just to export the new data into a clean csv from the dirty csv.

In [48]:
OUTPUT_PATH = 'Emergency_Triage_Dataset_Day2_MAP — Shari, Gabrielle, Asharah, & Josiah.csv'
df.to_csv(OUTPUT_PATH, index=False)
print(f'Saved → {OUTPUT_PATH}')

Saved → Emergency_Triage_Dataset_Day2_MAP — Shari, Gabrielle, Asharah, & Josiah.csv
